# Stage 2 — tools parser + combined 3B repair
Fresh raw input → intent contract → 3B code generation → verification / combined 3B FP16 repair → reference evaluation. Combined repair uses compact failure evidence, one bounded fresh-code recovery, and a declared-output-column guard. The three-proposal/tool/token limits are unchanged. Both generation and repair use the same pinned Qwen2.5-3B-Instruct model and full FP16 weights. Neither role uses 4-bit or 8-bit quantization. The existing intent-parser adapters retain their tested NF4 loading.

Inputs: **stage2_combined_3b_source.zip** and **tool_use_train_1_5b.zip**. This notebook is independent of the other method's outputs and all previous parser/Stage 2 result files. No training is performed. Use a GPU with at least 12 GiB memory; T4 is supported by the preflight. Only one model is resident at a time. With two GPUs the generator uses GPU 1; parser/repair use GPU 0.

Selective facts uses the corrected scope router plus operator normalization, without model correction. Tools uses the trained 1.5B tool adapter and its existing bounded loop, with normalization on contract replies. Every tool attempt/error, including placeholder queries, remains recorded. Both parser methods share the same generator, combined repair, schema guard, sandbox and limits. Parser prompts/adapters and generator prompts match the completed 3B control workflow. No parser correction or new facts routing is introduced.

Run a two-task infrastructure smoke, the full development split, then all 100 diagnostic test tasks. Each stage creates fresh predictions and a separate bundle. For a staged run, stop after the development cell and review its summary before running the final cell. A smoke can accept/reject/escalate: it checks infrastructure, not a quality threshold. The previously inspected test tasks provide diagnostic comparison, not independent research evidence. Download **stage2_combined_3b_tools_proposed.zip** and the executed notebook. Compare both methods using the CPU collector from this same pack.

**Kaggle:** attach both required ZIPs as private datasets, enable Internet and T4 GPU (T4 x2 also works), then Save Version → Run All.

Attach the new source ZIP plus the same selected adapter dataset used by the completed 3B run. The notebook verifies adapter hashes. Use an existing extracted/clean adapter dataset; avoid uploading full training archives containing nested baseline/checkpoint ZIPs.


In [1]:
import datetime,hashlib,json,os,shutil,subprocess,sys,zipfile
from pathlib import Path
INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working')
WORK.mkdir(parents=True,exist_ok=True)
RUN_ID=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR=WORK/('run_'+RUN_ID);RUN_DIR.mkdir()
DRIVE_RESULTS=None
def save_results():
    if DRIVE_RESULTS is not None:
        DRIVE_RESULTS.mkdir(parents=True,exist_ok=True)
        for p in RUN_DIR.iterdir():
            if p.is_file() and p.suffix in ['.zip','.log','.json','.txt']:shutil.copy2(p,DRIVE_RESULTS/p.name)
        print('Drive results:',DRIVE_RESULTS)
print('Run folder:',RUN_DIR)

METHOD='tools'
for name in ['stage2_combined_3b_source.zip','tool_use_train_1_5b.zip']:
    matches=list(INPUT.rglob(name));print(('found ZIP: ' if matches else 'No intact ZIP (may be extracted): ')+name,*matches)
print('The next cells verify source files and adapter contents, including automatically extracted datasets.')


Run folder: /kaggle/working/run_20261115_063625_186044
No intact ZIP (may be extracted): stage2_combined_3b_source.zip
No intact ZIP (may be extracted): tool_use_train_1_5b.zip
The next cells verify source files and adapter contents, including automatically extracted datasets.


In [2]:
MANIFEST_SHA='d9290a6ccb28b15d544a306a6d88f77a91bc84e455078a19e0b24e7bd4004ac2'
ROOT=WORK/('stage2_combined_3b_source_'+MANIFEST_SHA[:12])
def unpack(raw,read):
    assert hashlib.sha256(raw).hexdigest()==MANIFEST_SHA,'Wrong source ZIP version'
    manifest=json.loads(raw);assert manifest['experiment']=='stage2-combined-3b-fp16-v1'
    for name,sha in manifest['sha256'].items():
        relative=Path(name);assert not relative.is_absolute() and '..' not in relative.parts
        data=read(name);assert hashlib.sha256(data).hexdigest()==sha,f'Changed source/input: {name}'
        path=ROOT/relative;path.parent.mkdir(parents=True,exist_ok=True)
        if path.exists():assert path.read_bytes()==data,f'Changed working source: {name}'
        else:path.write_bytes(data)
    sidecar=ROOT/'_stage2_combined_3b_manifest.json'
    if sidecar.exists():assert sidecar.read_bytes()==raw,'Changed source manifest'
    else:sidecar.write_bytes(raw)
    print('Verified fresh-run source/data:',len(manifest['sha256']),'files',MANIFEST_SHA)
found=False
for path in INPUT.rglob('manifest.json'):
    if hashlib.sha256(path.read_bytes()).hexdigest()==MANIFEST_SHA:
        unpack(path.read_bytes(),lambda name:(path.parent/name).read_bytes());found=True;break
if not found:
    for path in INPUT.rglob('stage2_combined_3b_source.zip'):
        with zipfile.ZipFile(path) as z:
            assert len(z.namelist())==len(set(z.namelist()))
            prefix='stage2_combined_3b_source/';name=prefix+'manifest.json'
            if name not in z.namelist():continue
            raw=z.read(name)
            if hashlib.sha256(raw).hexdigest()!=MANIFEST_SHA:continue
            unpack(raw,lambda name:z.read(prefix+name));found=True;break
assert found,'Attach/place stage2_combined_3b_source.zip in the input folder.'
PACKAGES=WORK/'stage2_combined_3b_packages'
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(PACKAGES),'-r',str(ROOT/'requirements.txt')],check=True)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[ROOT,ROOT/'src',PACKAGES])),PYTHONUNBUFFERED='1',TOKENIZERS_PARALLELISM='false')


Verified fresh-run source/data: 272 files d9290a6ccb28b15d544a306a6d88f77a91bc84e455078a19e0b24e7bd4004ac2


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.33.3 requires PyYAML>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
ydata-profiling 4.18.4 requires PyYAML<6.1,>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
aiobotocore 2.26.0 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 3.0.0 which is incompatible.
botocore 1.41.5 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 3.0.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
d

In [3]:
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.20.2','peft==0.21.2','bitsandbytes==0.50.2','accelerate==1.15.0','tokenizers==0.23.3','safetensors==0.8.0','huggingface-hub==1.33.0'],check=True)
subprocess.run([sys.executable,'-m','eval.stage2_independent_v2.environment','--prepare'],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-c',"import torch; assert torch.cuda.is_available(); assert all(torch.cuda.get_device_properties(i).total_memory/2**30>=12 for i in range(torch.cuda.device_count())); print(torch.__version__,[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-m','finetune.selective_facts.attention'],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-m','eval.stage2_combined_3b.preflight'],cwd=ROOT,env=ENV,check=True)
path_file=RUN_DIR/'adapter_path.txt'
subprocess.run([sys.executable,'-m','eval.stage2_combined_3b.adapters','--method',METHOD,'--input',str(INPUT),'--work',str(WORK),'--out',str(path_file)],cwd=ROOT,env=ENV,check=True)
ADAPTER=Path(path_file.read_text());save_results()


Removing optional torchao 0.10.0: incompatible with PEFT 0.21.2; this workflow does not use torchao.
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
{"torchao": null, "action": "removed_incompatible", "previous_version": "0.10.0"}
{"probe": "cpu_lora_injection_and_forward", "trainable_parameters": 16, "status": "passed"}
2.11.0+cu130 ['Tesla T4', 'Tesla T4']
{"event": "attention_visible_devices", "devices": [0, 1], "cuda_visible_devices": null}
{"event": "attention_shape", "device": "cuda:0", "query_shape": [1, 4, 32, 128], "original_key_shape": [1, 2, 32, 128], "expanded_key_shape": [1, 4, 32, 128], "dtype": "torch.float16", "native_gqa_expanded": true, "stage": "prefill", "has_mask": false, "is_causal": true, "backend": "EFFICIENT_ATTENTION", "math_fallback_allowed": false}
{"event": "kernel_self_test_passed", "device": "cuda:0", "case": "causal_prefill", "max_absolute_error": 0.001953125, "rtol": 0.005, "atol": 0.005

In [4]:
DEV_COUNT=20
def execute(smoke,split='test'):
    name='stage2_combined_3b_'+METHOD+'_proposed'+('_smoke' if smoke else '_dev' if split=='dev' else '')
    out=RUN_DIR/name;assert not out.exists(),'Use a fresh run folder.'
    command=[sys.executable,'-m','eval.stage2_combined_3b.runner','--method',METHOD,'--preset','proposed','--adapter',str(ADAPTER),'--out',str(out),'--packages',str(PACKAGES),'--split',split]
    if smoke:command+=['--smoke']
    log=RUN_DIR/(name+'.log')
    try:
        with log.open('w') as f:
            process=subprocess.Popen(command,cwd=ROOT,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in process.stdout:
                print(line,end='');f.write(line);f.flush()
            code=process.wait()
    finally:save_results()
    assert code==0,f'Run failed: inspect {log} and the partial ZIP. A partial job is not a result.'
    assert json.loads((out/'completion.json').read_text())['complete']
    assert len(list((out/'tasks').glob('*.json')))==(2 if smoke else DEV_COUNT if split=='dev' else 100)
    return out.with_suffix('.zip')
SMOKE=execute(True)
DEV=execute(False,split='dev')
print('Development summary:',(DEV.with_suffix('')/'summary.json').read_text())
print('Smoke and end-to-end development completed. Inspect the development summary; the next cell runs 100 diagnostic test tasks freshly.')



Loading weights: 100%|██████████| 338/338 [00:02<00:00, 163.83it/s]
PARSE 1/2 adult_col_001: ok
PARSE 2/2 titanic_miss_004: ok

Fetching 2 files: 100%|██████████| 2/2 [00:31<00:00, 15.56s/it]

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 127.06it/s]
GENERATE 1/2 adult_col_001
GENERATE 2/2 titanic_miss_004

Loading weights: 100%|██████████| 434/434 [00:03<00:00, 127.31it/s]
DONE 1/2 adult_col_001: ACCEPT
DONE 2/2 titanic_miss_004: ACCEPT

Loading weights: 100%|██████████| 338/338 [00:02<00:00, 164.21it/s]
PARSE 1/20 adult_col_006: ok
PARSE 2/20 adult_conv_006: ok
PARSE 3/20 adult_dedup_006: ok
PARSE 4/20 adult_filt_006: ok
PARSE 5/20 adult_miss_006: ok
PARSE 6/20 airbnb_col_006: ok
PARSE 7/20 airbnb_conv_006: ok
PARSE 8/20 airbnb_dedup_006: ok
PARSE 9/20 airbnb_filt_006: ok
PARSE 10/20 airbnb_miss_006: ok
PARSE 11/20 taxi_col_006: ok
PARSE 12/20 taxi_conv_006: ok
PARSE 13/20 taxi_dedup_006: ok
PARSE 14/20 taxi_filt_006: ok
PARSE 15/20 taxi_miss_006: ok
PARSE 16/20 titanic_co

In [5]:
BUNDLE=execute(False)
from IPython.display import display,FileLink
display(FileLink(str(BUNDLE)))
if DRIVE_RESULTS is not None:print('Drive copy:',DRIVE_RESULTS/BUNDLE.name)
print('COMPLETE: download the full ZIP and this executed notebook. Run the other method independently, then the CPU collector.')



Loading weights: 100%|██████████| 338/338 [00:02<00:00, 161.87it/s]
PARSE 1/100 adult_col_001: ok
PARSE 2/100 adult_col_002: ok
PARSE 3/100 adult_col_003: ok
PARSE 4/100 adult_col_004: ok
PARSE 5/100 adult_col_005: ok
PARSE 6/100 adult_conv_001: ok
PARSE 7/100 adult_conv_002: ok
PARSE 8/100 adult_conv_003: ok
PARSE 9/100 adult_conv_004: ok
PARSE 10/100 adult_conv_005: ok
PARSE 11/100 adult_dedup_001: ok
PARSE 12/100 adult_dedup_002: ok
PARSE 13/100 adult_dedup_003: ok
PARSE 14/100 adult_dedup_004: ok
PARSE 15/100 adult_dedup_005: ok
PARSE 16/100 adult_filt_001: ok
PARSE 17/100 adult_filt_002: ok
PARSE 18/100 adult_filt_003: ok
PARSE 19/100 adult_filt_004: ok
PARSE 20/100 adult_filt_005: parser_invalid_response
PARSE 21/100 adult_miss_001: ok
PARSE 22/100 adult_miss_002: ok
PARSE 23/100 adult_miss_003: ok
PARSE 24/100 adult_miss_004: ok
PARSE 25/100 adult_miss_005: ok
PARSE 26/100 airbnb_col_001: ok
PARSE 27/100 airbnb_col_002: ok
PARSE 28/100 airbnb_col_003: ok
PARSE 29/100 airbnb_col

/kaggle/working/run_20261115_063625_186044/stage2_combined_3b_tools_proposed.zip

COMPLETE: download the full ZIP and this executed notebook. Run the other method independently, then the CPU collector.
